# PhysioTwin Model Lab

**Train, evaluate and export a temporal movement-quality classifier from pose landmarks.**

This notebook is the ML proof-of-work for the PhysioTwin hackathon prototype. It turns a sequence of 33 MediaPipe landmarks into normalized motion features, trains a Random Forest baseline, evaluates it using held-out movement sequences, and exports a model artifact plus a model card.

> Safety: this is a research prototype. Synthetic demo results are not clinical validation and the exported model must not diagnose, prescribe or replace a physiotherapist.

## Expected dataset

Upload `landmarks.csv` with one row per video frame:

- `sequence_id`: unique repetition/video identifier
- `frame`: increasing frame number
- `exercise`: e.g. `push-up`, `pull-up`, `mini-squat`
- `form_label`: e.g. `correct`, `hip_sag`, `incomplete_range`, `momentum`
- `x0,y0,z0,v0 ... x32,y32,z32,v32`: MediaPipe landmark coordinates and visibility

If no file is uploaded, the notebook creates a clearly marked synthetic dataset so the complete pipeline can be demonstrated.

In [ ]:
!pip -q install pandas scikit-learn seaborn joblib

import json, os, warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings('ignore')
print('PhysioTwin Model Lab ready')

In [ ]:
# Upload a real landmark CSV in Colab, or continue with the synthetic demo.
OUTPUT_DIR = Path('/content') if Path('/content').exists() else Path.cwd()
DATA_PATH = OUTPUT_DIR / 'landmarks.csv'
DEMO_MODE = not DATA_PATH.exists()

if DEMO_MODE:
    print('DEMO MODE: generating synthetic motion features. These metrics are not clinical evidence.')
else:
    print('REAL DATA MODE:', DATA_PATH)

In [ ]:
LANDMARKS = 33

def normalize_frame(frame):
    xyz = frame.reshape(LANDMARKS, 4).copy()
    hip_center = (xyz[23, :3] + xyz[24, :3]) / 2
    shoulder_center = (xyz[11, :3] + xyz[12, :3]) / 2
    torso = max(np.linalg.norm(shoulder_center - hip_center), 1e-6)
    xyz[:, :3] = (xyz[:, :3] - hip_center) / torso
    return xyz

def sequence_features(sequence):
    normalized = np.stack([normalize_frame(frame) for frame in sequence])
    coords = normalized[:, :, :3]
    velocity = np.diff(coords, axis=0, prepend=coords[:1])
    visibility = normalized[:, :, 3]
    return np.concatenate([
        coords.mean(axis=0).ravel(),
        coords.std(axis=0).ravel(),
        np.ptp(coords, axis=0).ravel(),
        np.abs(velocity).mean(axis=0).ravel(),
        visibility.mean(axis=0),
    ])

FEATURE_COUNT = LANDMARKS * 3 * 4 + LANDMARKS
print('Temporal features per sequence:', FEATURE_COUNT)

In [ ]:
def load_real_sequences(path):
    raw = pd.read_csv(path)
    required = {'sequence_id', 'frame', 'exercise', 'form_label'}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    landmark_cols = [f'{axis}{i}' for i in range(LANDMARKS) for axis in ('x','y','z','v')]
    missing_landmarks = set(landmark_cols) - set(raw.columns)
    if missing_landmarks:
        raise ValueError(f'Missing {len(missing_landmarks)} landmark columns')
    rows = []
    for sequence_id, group in raw.sort_values('frame').groupby('sequence_id'):
        sequence = group[landmark_cols].to_numpy(float)
        rows.append({
            'sequence_id': sequence_id,
            'exercise': group.exercise.iloc[0],
            'form_label': group.form_label.iloc[0],
            'features': sequence_features(sequence),
        })
    return pd.DataFrame(rows)

def synthetic_sequences(n_per_class=90):
    classes = ['push-up:correct', 'push-up:hip_sag', 'pull-up:correct',
               'pull-up:momentum', 'mini-squat:correct', 'mini-squat:shallow']
    rows = []
    for class_index, label in enumerate(classes):
        center = np.random.default_rng(SEED + class_index).normal(0, .28, FEATURE_COUNT)
        center[class_index * 18:(class_index + 1) * 18] += 1.6
        for sample in range(n_per_class):
            subject = sample % 18
            features = center + np.random.normal(0, .34, FEATURE_COUNT) + subject * .002
            exercise, form_label = label.split(':')
            rows.append({'sequence_id': f'{exercise}-{subject}-{sample}',
                         'exercise': exercise, 'form_label': form_label,
                         'features': features})
    return pd.DataFrame(rows)

dataset = synthetic_sequences() if DEMO_MODE else load_real_sequences(DATA_PATH)
dataset['target'] = dataset.exercise + ':' + dataset.form_label
print(dataset.target.value_counts())

In [ ]:
# Split by sequence/subject group to avoid frame leakage.
X = np.stack(dataset.features)
y = dataset.target.to_numpy()
groups = dataset.sequence_id.str.rsplit('-', n=2).str[0:2].str.join('-').to_numpy()

splitter = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups))

model = Pipeline([
    ('scale', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=320, max_depth=18, min_samples_leaf=2,
        class_weight='balanced', random_state=SEED, n_jobs=-1
    )),
])
model.fit(X[train_idx], y[train_idx])
pred = model.predict(X[test_idx])
balanced_acc = balanced_accuracy_score(y[test_idx], pred)
print(f'Balanced accuracy: {balanced_acc:.3f}')
print(classification_report(y[test_idx], pred, zero_division=0))

In [ ]:
labels = sorted(dataset.target.unique())
matrix = confusion_matrix(y[test_idx], pred, labels=labels, normalize='true')
plt.figure(figsize=(10, 7))
sns.heatmap(matrix, annot=True, fmt='.2f', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('PhysioTwin held-out sequence confusion matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.tight_layout(); plt.show()

In [ ]:
# Export the trained baseline and an auditable model card.
model_path = OUTPUT_DIR / 'physiotwin_temporal_rf.joblib'
card_path = OUTPUT_DIR / 'physiotwin_model_card.json'
joblib.dump(model, model_path)
model_card = {
    'name': 'PhysioTwin Temporal Random Forest',
    'version': '0.1-hackathon',
    'demo_mode': DEMO_MODE,
    'balanced_accuracy': round(float(balanced_acc), 4),
    'classes': labels,
    'feature_count': int(X.shape[1]),
    'train_sequences': int(len(train_idx)),
    'test_sequences': int(len(test_idx)),
    'intended_use': 'Research demonstration of exercise and form classification from pose sequences.',
    'not_for': ['diagnosis', 'autonomous treatment', 'pain assessment', 'emergency decisions'],
    'limitations': ['single-camera landmarks', 'occlusion sensitivity', 'requires clinician-labelled real data'],
}
card_path.write_text(json.dumps(model_card, indent=2))
print(json.dumps(model_card, indent=2))
print(f'\nArtifacts written to {model_path} and {card_path}')

## What to show the judges

1. Explain the landmark sequence dataset and why the split is by repetition/subject rather than frame.
2. Run the training cell and show balanced accuracy plus per-class precision/recall.
3. Show the confusion matrix to discuss which form errors are difficult to separate.
4. Open the exported model card and point out intended use, limitations and safety boundaries.
5. Replace demo data with clinician-labelled recordings before claiming real-world accuracy.